# Telco Customer Churn - Keşifsel Veri Analizi ve Model Eğitimi

Bu notebook'ta:
1. Veri setini tanıyacağız
2. Keşifsel veri analizi (EDA) yapacağız
3. Veriyi modele uygun hale getireceğiz
4. Birden fazla model deneyeceğiz
5. Hiperparametre optimizasyonu yapacağız
6. En iyi modeli seçip kaydedeceğiz

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Veri Setini Tanıma

In [ ]:
# CSV dosyasını okuyup df adli degiskene atiyoruz
df = pd.read_csv('../data/telco_customer_churn.csv')
print(f'Veri boyutu: {df.shape[0]} satir, {df.shape[1]} sutun')
df.head()

In [ ]:
# Her sutunun veri tipi ve kac dolu deger oldugu
df.info()

In [ ]:
# Istatistiksel ozet
df.describe()

In [ ]:
# Bos (null) deger kontrolu
print('Null deger sayilari:')
print(df.isnull().sum())
print(f'\nToplam null: {df.isnull().sum().sum()}')

## 2. Keşifsel Veri Analizi (EDA)

In [ ]:
# Hedef degisken dagilimi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='Churn', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Churn Dagilimi')
axes[0].set_xlabel('Musteri Ayrildi mi?')

churn_pct = df['Churn'].value_counts(normalize=True) * 100
axes[1].pie(churn_pct, labels=['Kaldi', 'Ayrildi'], autopct='%1.1f%%',
            colors=['#66b3ff', '#ff6666'], startangle=90)
axes[1].set_title('Churn Yuzdesel Dagilimi')

plt.tight_layout()
plt.show()

print(f'Kalan musteri: %{churn_pct["No"]:.1f}')
print(f'Ayrilan musteri: %{churn_pct["Yes"]:.1f}')
print(f'\nNot: Veri seti dengesiz (%{churn_pct["No"]:.0f} / %{churn_pct["Yes"]:.0f})')

In [ ]:
# Sayisal degiskenlerin dagilimi
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges'],
                           ['steelblue', 'salmon', 'green']):
    if col == 'TotalCharges':
        data = pd.to_numeric(df[col], errors='coerce')
    else:
        data = df[col]
    data.hist(ax=ax, bins=30, color=color, edgecolor='white')
    ax.set_title(f'{col} Dagilimi')
    ax.set_ylabel('Frekans')

plt.tight_layout()
plt.show()

In [ ]:
# Sayisal degiskenler - Churn bazli boxplot
df_temp = df.copy()
df_temp['TotalCharges'] = pd.to_numeric(df_temp['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(x='Churn', y=col, data=df_temp, ax=ax, palette='Set2')
    ax.set_title(f'{col} vs Churn')

plt.tight_layout()
plt.show()

print('Gozlemler:')
print('- tenure: Ayrilan musterilerin tenure degeri dusuk (yeni musteriler daha cok ayriliyor)')
print('- MonthlyCharges: Ayrilan musterilerin aylik ucreti daha yuksek')
print('- TotalCharges: Ayrilan musterilerin toplam ucreti dusuk (tenure ile tutarli)')

In [ ]:
# Kategorik degiskenler - Churn iliskisi
kategorik = ['Contract', 'PaymentMethod', 'InternetService', 'TechSupport',
              'OnlineSecurity', 'Partner', 'Dependents']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, col in zip(axes.flatten(), kategorik):
    sns.countplot(x=col, hue='Churn', data=df, ax=ax, palette='Set2')
    ax.set_title(f'{col} - Churn Iliskisi')
    ax.tick_params(axis='x', rotation=25)
    ax.legend(title='Churn', loc='upper right')

# Bos kalan alt plotlari kaldir
for i in range(len(kategorik), len(axes.flatten())):
    axes.flatten()[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Korelasyon matrisi (encoding oncesi sayisal degiskenler)
df_numeric = df_temp[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']].copy()
df_numeric['Churn'] = df_temp['Churn'].map({'Yes': 1, 'No': 0})

plt.figure(figsize=(8, 6))
sns.heatmap(df_numeric.corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Sayisal Degiskenler Korelasyon Matrisi')
plt.tight_layout()
plt.show()

## 3. Veri On Isleme (Preprocessing)

In [ ]:
# TotalCharges sutunu sayi olmasi gerekirken metin olarak gelmis
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'TotalCharges null sayisi: {df["TotalCharges"].isnull().sum()}')

# Null degerleri olan 11 satiri siliyoruz
df = df.dropna(subset=['TotalCharges'])
print(f'Temizlik sonrasi veri boyutu: {df.shape}')

In [ ]:
# customerID sutunu isimize yaramaz, kaldiriyoruz
df = df.drop(columns=['customerID'])

# Hedef degiskeni encode et
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Binary encoding: Yes/No -> 1/0
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

binary_cols = ['Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0,
                           'No internet service': 0,
                           'No phone service': 0})

print('Binary encoding tamamlandi')
print(f'Kalan kategorik sutunlar: {df.select_dtypes(include="object").columns.tolist()}')

In [ ]:
# One-Hot Encoding: Birden fazla kategorisi olan sutunlar
df = pd.get_dummies(df, columns=['MultipleLines', 'InternetService',
                                  'Contract', 'PaymentMethod'], drop_first=True)

# Boolean -> int donusumu
df = df.astype({col: int for col in df.select_dtypes(include='bool').columns})

print(f'Final feature sayisi: {df.shape[1] - 1}')
print(f'Final veri boyutu: {df.shape}')
df.head()

## 4. Model Egitimi

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X = input features, y = hedef degisken
X = df.drop(columns=['Churn'])
y = df['Churn']
feature_names = X.columns.tolist()

print(f'X shape: {X.shape}')
print(f'y dagilimi:\n{y.value_counts()}')
print(f'\nAyrilma orani: %{(y == 1).sum() / len(y) * 100:.1f}')

In [ ]:
# Veriyi egitim ve test olarak bolme (%80 / %20)
# stratify=y: sinif dagilimini korur
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Egitim: {X_train.shape}')
print(f'Test: {X_test.shape}')

In [ ]:
# Olceklendirme: Sayisal sutunlari ayni olcege getiriyoruz
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Olceklendirme tamamlandi!')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

# 3 farkli algoritma deniyoruz
modeller = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0)
}

sonuclar = {}

for isim, model in modeller.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Olasilik skorlari
    
    sonuclar[isim] = {
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4)  # predict_proba ile hesapla!
    }
    
    print(f'\n{"="*50}')
    print(f'{isim}')
    print(f'{"="*50}')
    print(f'Accuracy: {sonuclar[isim]["Accuracy"]}')
    print(f'F1 Score: {sonuclar[isim]["F1 Score"]}')
    print(f'ROC-AUC:  {sonuclar[isim]["ROC-AUC"]}')
    print(f'\nConfusion Matrix:')
    cm = confusion_matrix(y_test, y_pred)
    print(f'  TN={cm[0][0]:4d}  FP={cm[0][1]:4d}')
    print(f'  FN={cm[1][0]:4d}  TP={cm[1][1]:4d}')
    print(f'\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['Kalan', 'Ayrilan']))

sonuc_df = pd.DataFrame(sonuclar).T
print(f'\n{"="*50}')
print('MODEL KARSILASTIRMA')
print(f'{"="*50}')
print(sonuc_df.to_string())

## 5. Sinif Dengesizligi Cozumleri

Veri setimizde %73/%27 oraninda dengesizlik var. Bunu asmak icin iki yontem deneyecegiz:
- `class_weight='balanced'`: Modelin az temsil edilen sinifa daha fazla onem vermesini saglar
- SMOTE: Azinlik sinifindan sentetik ornekler uretir

In [ ]:
# class_weight='balanced' ile deneme
print('class_weight="balanced" sonuclari:')
print('-' * 60)

balanced_models = {
    'LR Balanced': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'RF Balanced': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
}

for isim, model in balanced_models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    print(f'{isim}: Accuracy={accuracy_score(y_test, y_pred):.4f}, '
          f'F1={f1_score(y_test, y_pred):.4f}, ROC-AUC={roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
from imblearn.over_sampling import SMOTE

# SMOTE ile deneme
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
print(f'SMOTE sonrasi egitim boyutu: {X_train_sm.shape}')
print(f'Sinif dagilimi: {np.bincount(y_train_sm)} (dengelendi!)')
print('-' * 60)

smote_models = {
    'LR+SMOTE': LogisticRegression(max_iter=1000, random_state=42),
    'RF+SMOTE': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGB+SMOTE': XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
}

for isim, model in smote_models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    print(f'{isim}: Accuracy={accuracy_score(y_test, y_pred):.4f}, '
          f'F1={f1_score(y_test, y_pred):.4f}, ROC-AUC={roc_auc_score(y_test, y_proba):.4f}')

## 6. Hiperparametre Optimizasyonu (GridSearchCV)

Her model icin en iyi parametreleri bulmak icin 5-fold cross-validation ile GridSearch uyguluyoruz.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression GridSearch
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced'],
    'max_iter': [1000]
}

grid_lr = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid_lr, cv=cv, scoring='roc_auc', n_jobs=-1
)
grid_lr.fit(X_train_scaled, y_train)

print('Logistic Regression GridSearch')
print(f'En iyi parametreler: {grid_lr.best_params_}')
print(f'En iyi CV ROC-AUC: {grid_lr.best_score_:.4f}')
y_proba_lr = grid_lr.predict_proba(X_test_scaled)[:, 1]
y_pred_lr = grid_lr.predict(X_test_scaled)
print(f'Test ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')
print(f'Test F1: {f1_score(y_test, y_pred_lr):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')

In [ ]:
# XGBoost GridSearch
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
}

grid_xgb = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
    param_grid_xgb, cv=cv, scoring='roc_auc', n_jobs=-1
)
grid_xgb.fit(X_train_scaled, y_train)

print('XGBoost GridSearch')
print(f'En iyi parametreler: {grid_xgb.best_params_}')
print(f'En iyi CV ROC-AUC: {grid_xgb.best_score_:.4f}')
y_proba_xgb = grid_xgb.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb = grid_xgb.predict(X_test_scaled)
print(f'Test ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}')
print(f'Test F1: {f1_score(y_test, y_pred_xgb):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}')

In [ ]:
# Random Forest GridSearch
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'class_weight': [None, 'balanced']
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf, cv=cv, scoring='roc_auc', n_jobs=-1
)
grid_rf.fit(X_train_scaled, y_train)

print('Random Forest GridSearch')
print(f'En iyi parametreler: {grid_rf.best_params_}')
print(f'En iyi CV ROC-AUC: {grid_rf.best_score_:.4f}')
y_proba_rf = grid_rf.predict_proba(X_test_scaled)[:, 1]
y_pred_rf = grid_rf.predict(X_test_scaled)
print(f'Test ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'Test F1: {f1_score(y_test, y_pred_rf):.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')

## 7. En Iyi Modeli Secme ve Feature Importance

In [ ]:
# Tum GridSearch sonuclarini karsilastir
all_results = {
    'LR GridSearch': roc_auc_score(y_test, y_proba_lr),
    'XGB GridSearch': roc_auc_score(y_test, y_proba_xgb),
    'RF GridSearch': roc_auc_score(y_test, y_proba_rf),
}

print('TUM MODEL SONUCLARI (GridSearchCV sonrasi):')
print('=' * 50)
for isim, score in sorted(all_results.items(), key=lambda x: x[1], reverse=True):
    print(f'{isim}: ROC-AUC = {score:.4f}')

best_name = max(all_results, key=all_results.get)
grids = {'LR GridSearch': grid_lr, 'XGB GridSearch': grid_xgb, 'RF GridSearch': grid_rf}
best_model = grids[best_name].best_estimator_
print(f'\nEN IYI MODEL: {best_name} (ROC-AUC: {all_results[best_name]:.4f})')

In [ ]:
# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importances = np.abs(best_model.coef_[0])

feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True).tail(15)

plt.figure(figsize=(10, 6))
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color='steelblue')
plt.title('En Onemli 15 Feature')
plt.xlabel('Onem Skoru')
plt.tight_layout()
plt.show()

print(feat_imp.sort_values('Importance', ascending=False).to_string(index=False))

## 8. Modeli Kaydetme

In [ ]:
import joblib

# En iyi modeli ve ilgili dosyalari kaydet
joblib.dump(best_model, '../models/model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(feature_names, '../models/feature_names.pkl')

print(f'Model kaydedildi: {type(best_model).__name__}')
print(f'ROC-AUC: {all_results[best_name]:.4f}')
print(f'\nKaydedilen dosyalar:')
print(f'  - models/model.pkl')
print(f'  - models/scaler.pkl')
print(f'  - models/feature_names.pkl')